# SecureMailScope — model trainingTrains the two models the pipeline uses and produces the numbers for the metrics slide.**Why this runs here and not on the dev machines:** Windows Smart App Control blocks numpy's compiledextension, so scikit-learn cannot import locally (ADR-0012). Colab has no such restriction.**You need one file:** `data/corpus.csv`, produced by```python -m securemailscope.ml.train --samples 10000 --corpus-only```It is self-sufficient — 51 feature columns, the rule-engine label, and a precomputed `baseline_risk`per row, so nothing from the repo needs to be installed here.**Run every cell in order.** Takes about three minutes. The last cell downloads two `.joblib` filesthat go in `models/` in the repo.

In [ ]:
!pip install -q scikit-learn shap joblib pandas matplotlibimport sklearn, shapprint("scikit-learn", sklearn.__version__, "| shap", shap.__version__)

## 1. Upload `data/corpus.csv`

In [ ]:
from google.colab import filesuploaded = files.upload()          # choose data/corpus.csvimport pandas as pd, numpy as np, ioname = next(iter(uploaded))df = pd.read_csv(io.BytesIO(uploaded[name]))print(f"{len(df):,} rows x {df.shape[1]} columns")df.head(3)

In [ ]:
META = ["port", "port_role", "archetype", "label", "target", "baseline_risk"]FEATURES = [c for c in df.columns if c not in META]print(f"{len(FEATURES)} feature columns")SEVERITY_RANK = {"info": 0, "low": 1, "medium": 2, "high": 3, "critical": 4}X = df[FEATURES].astype(float).valuesy = df["label"].map(SEVERITY_RANK).astype(int).valuesprint("\nlabel distribution:")print(df["label"].value_counts(normalize=True).mul(100).round(1).astype(str) + " %")print("\narchetype distribution:")print(df["archetype"].value_counts(normalize=True).mul(100).round(1).astype(str) + " %")

## 2. SplitIndices are split alongside the arrays so the per-row `baseline_risk` follows the same rows.`train_test_split` shuffles — slicing the dataframe by position afterwards gives a different set,and one that overlaps training data. That bug silently flatters the model.

In [ ]:
from sklearn.model_selection import train_test_splitidx = np.arange(len(df))X_tr, X_te, y_tr, y_te, idx_tr, idx_te = train_test_split(    X, y, idx, test_size=0.25, random_state=42, stratify=y)print(f"train {len(X_tr):,}   test {len(X_te):,}")

## 3. Risk classifier (D16)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifierfrom sklearn.metrics import (accuracy_score, classification_report,                             confusion_matrix, roc_auc_score)clf = GradientBoostingClassifier(n_estimators=300, learning_rate=0.08,                                 max_depth=4, random_state=42)clf.fit(X_tr, y_tr)y_pred = clf.predict(X_te)proba  = clf.predict_proba(X_te)metrics = {    "accuracy":   round(float(accuracy_score(y_te, y_pred)), 4),    "roc_auc_ovr": round(float(roc_auc_score(y_te, proba, multi_class="ovr")), 4),    "n_train": int(len(X_tr)), "n_test": int(len(X_te)),}print(metrics)print()print(classification_report(y_te, y_pred, labels=[0,1,2,3,4],      target_names=["info","low","medium","high","critical"], zero_division=0))

In [ ]:
import matplotlib.pyplot as pltnames = ["info","low","medium","high","critical"]cm = confusion_matrix(y_te, y_pred, labels=[0,1,2,3,4])fig, ax = plt.subplots(figsize=(5.5, 4.6))ax.imshow(cm, cmap="Blues")ax.set_xticks(range(5), names, rotation=45, ha="right")ax.set_yticks(range(5), names)ax.set_xlabel("predicted"); ax.set_ylabel("actual")ax.set_title("Risk classifier — confusion matrix")for i in range(5):    for j in range(5):        ax.text(j, i, cm[i, j], ha="center", va="center",                color="white" if cm[i, j] > cm.max() * 0.5 else "black", fontsize=9)plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=200); plt.show()

## 4. Does it beat the baseline?The honest yardstick. `baseline_risk` is the rule-derived scorer the pipeline falls back to.**If the model does not beat it, do not claim it does** — say the baseline is what ships.

In [ ]:
ranks = np.array([int(c) for c in clf.classes_], dtype=float) / 4.0model_risk    = (proba * ranks).sum(axis=1)baseline_risk = df["baseline_risk"].values[idx_te]target        = df["target"].values[idx_te]metrics["model_mae"]    = round(float(np.abs(model_risk - target).mean()), 4)metrics["baseline_mae"] = round(float(np.abs(baseline_risk - target).mean()), 4)better = metrics["model_mae"] < metrics["baseline_mae"]gain = (1 - metrics["model_mae"] / metrics["baseline_mae"]) * 100print(f"baseline MAE  {metrics['baseline_mae']}")print(f"model    MAE  {metrics['model_mae']}")print()print(f"model beats baseline by {gain:.1f}%" if better else      "WARNING: the model does NOT beat the baseline. Do not claim it does.")

## 5. Anomaly detector (D17)Fitted on the **healthy archetypes of the training split only**. Fitting on everything teaches itthat compromised sessions are normal — the same mistake the fleet baseline used to make.

In [ ]:
from sklearn.ensemble import IsolationForestHEALTHY = {"modern", "standard", "dated"}arch_tr = df["archetype"].values[idx_tr]arch_te = df["archetype"].values[idx_te]healthy_tr = X_tr[np.isin(arch_tr, list(HEALTHY))]print(f"fitting on {len(healthy_tr):,} healthy training rows")iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)iso.fit(healthy_tr)is_anom = ~np.isin(arch_te, list(HEALTHY))scores  = -iso.decision_function(X_te)        # higher = strangerk = max(1, int(0.10 * len(scores)))top_k = np.argsort(scores)[::-1][:k]metrics["anomaly_roc_auc"] = round(float(roc_auc_score(is_anom, scores)), 4)metrics["anomaly_precision_at_10pct"] = round(float(is_anom[top_k].mean()), 4)metrics["anomaly_base_rate"] = round(float(is_anom.mean()), 4)print(f"ROC-AUC              {metrics['anomaly_roc_auc']}")print(f"precision @ top 10%  {metrics['anomaly_precision_at_10pct']}")print(f"base rate            {metrics['anomaly_base_rate']}")print(f"lift over random     {metrics['anomaly_precision_at_10pct']/metrics['anomaly_base_rate']:.1f}x")

## 6. SHAP — what the model actually learned

In [ ]:
explainer = shap.TreeExplainer(clf)sample = X_te[:400]sv = explainer.shap_values(sample)vals = np.abs(np.array(sv)).mean(axis=0) if isinstance(sv, list) else np.abs(sv)while vals.ndim > 1:    vals = vals.mean(axis=-1) if vals.shape[-1] != len(FEATURES) else vals.mean(axis=0)order = np.argsort(vals)[::-1][:15]plt.figure(figsize=(7, 5.5))plt.barh([FEATURES[i] for i in order][::-1], vals[order][::-1], color="#0f5fb5")plt.xlabel("mean |SHAP value|"); plt.title("Top 15 features driving risk")plt.tight_layout(); plt.savefig("shap_importance.png", dpi=200); plt.show()for i in order[:10]:    print(f"  {vals[i]:.4f}  {FEATURES[i]}")

## 7. SaveBundle format matches exactly what `RiskModel.load()` expects — do not rename the keys.

In [ ]:
import joblib, json, osos.makedirs("models", exist_ok=True)joblib.dump({"model": clf, "explainer": explainer,             "feature_names": FEATURES, "version": "1.0.0"},            "models/risk_classifier.joblib")joblib.dump({"model": iso, "feature_names": FEATURES, "version": "1.0.0"},            "models/anomaly_detector.joblib")with open("models/metrics.json", "w") as fh:    json.dump(metrics, fh, indent=2)print(json.dumps(metrics, indent=2))

In [ ]:
from google.colab import filesfor f in ["models/risk_classifier.joblib", "models/anomaly_detector.joblib",          "models/metrics.json", "confusion_matrix.png", "shap_importance.png"]:    files.download(f)

## 8. Back in the repoPut `risk_classifier.joblib`, `anomaly_detector.joblib` and `metrics.json` into `models/`, then:```python scripts/analyse.py testbed/out/fleet.pcap --out out/ --trust testbed/certs/_ca.derpython scripts/audit.py```The report header should now read **`risk backend: gradient_boosting`** instead of `rule_baseline`,and D16/D17 should move from PARTIAL to MET in the audit.`models/` is gitignored — the `.joblib` files are build artifacts. Share them with the teamdirectly, and keep `metrics.json` for the slide.